# Parquet → dataframe or CSV for the borefield visualization export

Reads the four committed parquet tables in `export/` (written by
`Borefield_visualization_export.ipynb`) — `ground_properties`, `boreholes`,
`borehole_segments`, `segment_heat_rates` — into four separate dataframes,
displays each table for review, suggests visualizations, and (optionally)
writes five files to `export_csv/`: the four table CSVs plus a copy of
`README.md` (the data dictionary), matching the original notebook's output.

**Why the CSVs are not committed:** at hourly aggregation
`segment_heat_rates.csv` is ~294 MB, which exceeds GitHub's hard 100 MB
per-file push limit — pushes containing it are rejected. The `export_csv/`
folder is therefore git-ignored on purpose; its contents are intended for
local analysis or out-of-band sharing (e.g. a OneDrive/SharePoint link or a
GitHub Release asset), not for committing.

**Why one parquet file per table (rather than one combined file):** parquet
holds a single schema per file, and the four tables have different columns —
combining them would force a superset schema with a table-discriminator
column and nulls, breaking dtypes and confusing readers. Separate files also
let you load only the table you need (the geometry tables are KB-scale; the
heat-rates table is tens of MB).

If you only need the data in Python, skip this notebook entirely:

```python
import pandas as pd
df = pd.read_parquet('export/segment_heat_rates.parquet')  # requires pyarrow
```

In [ ]:
"""Step 1 — Check / install pyarrow (reads the parquet files)."""
import sys, subprocess, importlib
from pathlib import Path

_LIB_TARGET = Path.home() / '.local' / 'impact_libs'

def _ensure_path():
    target = str(_LIB_TARGET)
    if target not in sys.path:
        sys.path.insert(0, target)

def _lib_ok(name):
    _ensure_path()
    try:
        importlib.import_module(name)
        return True
    except ImportError:
        return False

if _lib_ok('pyarrow'):
    print('pyarrow library is available.')
else:
    print(f'Installing pyarrow into {_LIB_TARGET} ...')
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--target', str(_LIB_TARGET), 'pyarrow'],
        capture_output=True, text=True)
    if r.returncode == 0:
        _ensure_path()
        print('Installed. If import fails below, restart the kernel and re-run this cell.')
    else:
        print(r.stderr)
        raise RuntimeError('Failed to install pyarrow')

pyarrow library is available.


In [ ]:
import shutil
from pathlib import Path

import pandas as pd

# --- Configuration --------------------------------------------------------
# Folder holding the committed parquet tables (default: the export next to
# this notebook).
EXPORT_DIR = Path.cwd() / 'export_parquet'

# Write CSV copies? Set False to only load/review the dataframes.
WRITE_CSV = True

# Where to write the CSVs (git-ignored: segment_heat_rates.csv is ~294 MB at
# hourly aggregation, over GitHub's 100 MB per-file push limit).
CSV_DIR = Path.cwd() / 'export_csv'

parquet_files = sorted(EXPORT_DIR.glob('*.parquet'))
if not parquet_files:
    raise FileNotFoundError(f'No .parquet files found in {EXPORT_DIR}')
print(f'Input : {EXPORT_DIR}  ({len(parquet_files)} parquet file(s))')
print(f'Output: {CSV_DIR}  (WRITE_CSV={WRITE_CSV})')

Input : /home/jovyan/impact/local_projects/Impact_collab_sg-kul_codingdev/PostProcess/Resources/borefield_visualization_export/export  (4 parquet file(s))
Output: /home/jovyan/impact/local_projects/Impact_collab_sg-kul_codingdev/PostProcess/Resources/borefield_visualization_export/export_csv  (WRITE_CSV=True)


## Import each parquet file as a dataframe and review

Each parquet file is loaded into its own dataframe (`tables[<name>]`) and
displayed below so the inputs can be reviewed: small tables in full, the
heat-rates table as a preview plus a numeric summary.

In [ ]:
tables = {}
for pq_path in parquet_files:
    name = pq_path.stem
    tables[name] = pd.read_parquet(pq_path)
    print(f'{pq_path.name:32s} {tables[name].shape[0]:>9,} rows x '
          f'{tables[name].shape[1]} cols  ({pq_path.stat().st_size / 1e6:,.2f} MB)')

borehole_segments.parquet              240 rows x 4 cols  (0.00 MB)
boreholes.parquet                       60 rows x 10 cols  (0.01 MB)
ground_properties.parquet                1 rows x 6 cols  (0.00 MB)


segment_heat_rates.parquet       2,102,400 rows x 10 cols  (16.80 MB)


In [ ]:
from IPython.display import display, Markdown

for name, df in tables.items():
    display(Markdown(f'### `{name}` — {df.shape[0]:,} rows x {df.shape[1]} cols'))
    # Small tables in full; the large heat-rates table as head + numeric summary.
    if len(df) <= 240:
        display(df)
    else:
        display(df.head(10))
        display(Markdown('Numeric summary:'))
        display(df.describe())

### `borehole_segments` — 240 rows x 4 cols

,borehole_id,segment_id,z_start_m,z_end_m
0,BH001,1,1.0,39.1
1,BH001,2,39.1,77.2
2,BH001,3,77.2,115.3
3,BH001,4,115.3,153.4
4,BH002,1,1.0,39.1
...,...,...,...,...
235,BH059,4,115.3,153.4
236,BH060,1,1.0,39.1
237,BH060,2,39.1,77.2
238,BH060,3,77.2,115.3


### `boreholes` — 60 rows x 10 cols

,borehole_id,zone_id,x_m,y_m,H_m,D_m,r_b_m,tilt_deg,orientation_deg,n_segments
0,BH001,Zone_01,0.000,0.000,152.4,1.0,0.0762,0.0,0.0,4
1,BH002,Zone_01,6.096,0.000,152.4,1.0,0.0762,0.0,0.0,4
2,BH003,Zone_01,12.192,0.000,152.4,1.0,0.0762,0.0,0.0,4
3,BH004,Zone_01,18.288,0.000,152.4,1.0,0.0762,0.0,0.0,4
4,BH005,Zone_02,0.000,6.096,152.4,1.0,0.0762,0.0,0.0,4
5,BH006,Zone_02,6.096,6.096,152.4,1.0,0.0762,0.0,0.0,4
6,BH007,Zone_02,12.192,6.096,152.4,1.0,0.0762,0.0,0.0,4
7,BH008,Zone_02,18.288,6.096,152.4,1.0,0.0762,0.0,0.0,4
8,BH009,Zone_03,0.000,12.192,152.4,1.0,0.0762,0.0,0.0,4
9,BH010,Zone_03,6.096,12.192,152.4,1.0,0.0762,0.0,0.0,4


### `ground_properties` — 1 rows x 6 cols

,T_undisturbed_K,gradient_K_m,z_start_gradient,k_soil_W_mK,rho_soil_kg_m3,c_soil_J_kgK
0,284.2611,0.0,10.0,3.513392,2400.0,1089.154


### `segment_heat_rates` — 2,102,400 rows x 10 cols

,time_s,time_end_s,borehole_id,segment_id,zone_id,q_segment_J,q_segment_W_avg,T_fluid_in_zone_K,T_fluid_out_zone_K,m_flow_zone_kg_s
0,0.0,3600.0,BH001,1,Zone_01,-6.182428e+05,-171.734123,283.571383,283.928595,2.885687
1,0.0,3600.0,BH002,1,Zone_01,-6.182428e+05,-171.734123,283.571383,283.928595,2.885687
2,0.0,3600.0,BH003,1,Zone_01,-6.182428e+05,-171.734123,283.571383,283.928595,2.885687
3,0.0,3600.0,BH004,1,Zone_01,-6.182428e+05,-171.734123,283.571383,283.928595,2.885687
4,3600.0,7200.0,BH001,1,Zone_01,-1.336880e+06,-371.355434,282.764115,283.435977,2.885521
5,3600.0,7200.0,BH002,1,Zone_01,-1.336880e+06,-371.355434,282.764115,283.435977,2.885521
6,3600.0,7200.0,BH003,1,Zone_01,-1.336880e+06,-371.355434,282.764115,283.435977,2.885521
7,3600.0,7200.0,BH004,1,Zone_01,-1.336880e+06,-371.355434,282.764115,283.435977,2.885521
8,7200.0,10800.0,BH001,1,Zone_01,-1.497221e+06,-415.894759,282.559667,283.201958,2.885560
9,7200.0,10800.0,BH002,1,Zone_01,-1.497221e+06,-415.894759,282.559667,283.201958,2.885560


Numeric summary:

,time_s,time_end_s,segment_id,q_segment_J,q_segment_W_avg,T_fluid_in_zone_K,T_fluid_out_zone_K,m_flow_zone_kg_s
count,2.102400e+06,2.102400e+06,2.102400e+06,2.102400e+06,2.102400e+06,2.102400e+06,2.102400e+06,2.102400e+06
mean,1.576620e+07,1.576980e+07,2.500000e+00,1.588835e+05,4.413432e+01,2.845986e+02,2.845288e+02,2.573514e+00
std,9.103661e+06,9.103661e+06,1.118034e+00,1.443232e+06,4.008979e+02,2.515697e+00,1.912267e+00,1.755387e-01
min,0.000000e+00,3.600000e+03,1.000000e+00,-4.004407e+06,-1.112335e+03,2.783509e+02,2.799049e+02,2.313508e+00
25%,7.883100e+06,7.886700e+06,1.750000e+00,-6.509871e+05,-1.808297e+02,2.827984e+02,2.830633e+02,2.410604e+00
50%,1.576620e+07,1.576980e+07,2.500000e+00,1.316556e+05,3.657101e+01,2.842349e+02,2.842681e+02,2.557320e+00
75%,2.364930e+07,2.365290e+07,3.250000e+00,6.968497e+05,1.935694e+02,2.861846e+02,2.859002e+02,2.731539e+00
max,3.153240e+07,3.153600e+07,4.000000e+00,8.718684e+06,2.421857e+03,2.961849e+02,2.926815e+02,2.887447e+00


## Suggested visualizations

Ideas per table (dataframes are in `tables[...]`; join on `borehole_id` /
`segment_id` / `zone_id` as needed):

1. **Plan-view heat map of annual net energy per borehole** — sum
   `q_segment_J` per `borehole_id` from `segment_heat_rates`, plot at (x, y)
   from `boreholes` as a color-coded scatter/grid. Shows which zones drive
   ground loading and edge-vs-interior imbalance.
2. **Depth-resolved seasonal profile** — heat map of month (x) vs the four
   segment depth bands from `borehole_segments` (y), colored by mean
   `q_segment_W_avg`. Shows how load shifts vertically through the year;
   feeds the ground-temperature contour work.
3. **Annual load duration curve per zone** — sorted hourly `q_segment_W_avg`
   summed per `zone_id`. Compares peak vs base loading across zones.
4. **Fluid temperature envelope** — monthly min/mean/max band of
   `T_fluid_in_zone_K` / `T_fluid_out_zone_K` per zone against design limits
   (key check for heat pump entering-water-temperature compliance).
5. **Cumulative ground energy balance** — running sum of `q_segment_J`
   field-wide over the year; quantifies annual imbalance and expected
   multi-year ground temperature drift (compare against
   `ground_properties` undisturbed temperature).

Outputs should be reviewed by the project engineer before external use.

## Optional: write five files to `export_csv/`

The four table CSVs plus a copy of `README.md` (data dictionary), matching
the original notebook's five-file output. Skipped when `WRITE_CSV = False`.

In [ ]:
if not WRITE_CSV:
    print('WRITE_CSV = False -> skipping CSV export.')
else:
    CSV_DIR.mkdir(parents=True, exist_ok=True)

    for name, df in tables.items():
        csv_path = CSV_DIR / f'{name}.csv'
        df.to_csv(csv_path, index=False)
        print(f'{csv_path.name:32s} {df.shape[0]:>9,} rows x {df.shape[1]} cols  '
              f'({csv_path.stat().st_size / 1e6:,.2f} MB)')

    readme_src = EXPORT_DIR / 'README.md'
    if readme_src.exists():
        shutil.copy2(readme_src, CSV_DIR / 'README.md')
        print(f'{"README.md":32s} data dictionary (copied)')
    else:
        print('README.md not found in export/; skipped copy.')

    big = [p for p in CSV_DIR.glob('*.csv') if p.stat().st_size > 100e6]
    if big:
        names = ', '.join(p.name for p in big)
        print(f'\nNOTE: {names} exceeds GitHub\'s 100 MB per-file limit and cannot be\n'
              'pushed; export_csv/ is git-ignored in this repo. Share out-of-band if\n'
              'needed (OneDrive/SharePoint link or a GitHub Release asset).')

borehole_segments.csv                  240 rows x 4 cols  (0.00 MB)
boreholes.csv                           60 rows x 10 cols  (0.00 MB)
ground_properties.csv                    1 rows x 6 cols  (0.00 MB)


segment_heat_rates.csv           2,102,400 rows x 10 cols  (272.61 MB)
README.md                        data dictionary (copied)

NOTE: segment_heat_rates.csv exceeds GitHub's 100 MB per-file limit and cannot be
pushed; export_csv/ is git-ignored in this repo. Share out-of-band if
needed (OneDrive/SharePoint link or a GitHub Release asset).
